# 예제 02. 컨텍스트 한계와 비용 · Example 02 — Context Limits and Cost

전산실험 2 · AI Agent 만들기 · **2주차** · 예상 소요 40분
Computational Experiments 2 · Building AI Agents · **Week 2** · about 40 minutes

## 목표 · Objectives
- **입력과 출력이 같은 예산을 나눠 쓴다**는 것을 실험으로 확인한다
- `finish_reason` 을 읽어 "잘린 응답" 을 구분한다
- 컨텍스트 한계까지 밀어 보고, **어떻게 실패하는지** 기록한다
- 에이전트 루프의 누적 토큰이 **왜 제곱으로 늘어나는지** 계산하고 실측한다
- 예산을 코드로 강제하는 `TokenBudget` 을 만든다

> **EN**
> - Confirm by experiment that **input and output share one budget**
> - Read `finish_reason` to tell a truncated answer apart from a finished one
> - Push to the context limit and record **how it fails**
> - Work out **why an agent loop's cumulative tokens grow quadratically**, then measure it
> - Build `TokenBudget`, which enforces the budget in code

> ⚠️ **서버를 배려하십시오.** 이 노트북은 긴 요청을 여러 번 보냅니다.
> 20명이 동시에 돌리면 서버가 느려집니다. 셀 사이에 넣어 둔 대기를
> 지우지 마세요. 전체 호출은 60회 남짓입니다.
>
> ⚠️ **EN — be considerate of the server.** This notebook sends long requests
> repeatedly, and it slows down when 20 people run it at once. Do not remove the
> waits placed between the cells. About 60 calls in total.

### 준비 · Setup

예제 01 과 같은 설정입니다. 여기에 예제 01 에서 만든 근사식을
기본 계수로 넣어 두었습니다. 자기 값으로 바꿔도 됩니다.

> **EN** — The same setup as Example 01, with the approximation you fitted there
> supplied as default coefficients. Substitute your own values if you like.

In [ ]:
!pip install -q openai

import os
from openai import OpenAI

def get_secret(name, secret=True):
    # Colab Secrets -> 환경변수 -> 직접 입력. 값은 코드에 적지 않는다.
    # EN — Colab Secrets -> environment variable -> prompt. Never write the value here.
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v:
            return v.strip()
    except Exception:
        pass
    if os.environ.get(name):
        return os.environ[name].strip()
    if secret:
        import getpass
        v = getpass.getpass(f"{name}: ").strip()
    else:
        v = input(f"{name}: ").strip()
    os.environ[name] = v
    return v


def get_base_url():
    url = get_secret("KMU_BASE_URL", secret=False).rstrip("/")
    for suffix in ("/chat/completions", "/completions"):
        if url.endswith(suffix):
            url = url[: -len(suffix)]
    return url


BASE_URL = get_base_url()
MODEL    = "gemma4"

client = OpenAI(api_key=get_secret("KMU_API_KEY"), base_url=BASE_URL, timeout=120)


def raw(messages, **kw):
    """응답 객체를 그대로 돌려준다. usage 를 봐야 하므로.

    EN — Return the whole response object, because we need to read `usage`.
    """
    kw.setdefault("temperature", 0.0)
    return client.chat.completions.create(model=MODEL, messages=messages, **kw)


def ask(prompt, system=None, **kw):
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    return raw(messages, **kw).choices[0].message.content.strip()

print("연결 준비 완료:", MODEL)

In [ ]:
import time

def 한글수(text):
    # EN — Count the Hangul syllable characters.
    return sum(1 for ch in text if "가" <= ch <= "힣")


def estimate_tokens(text, 절편=6.0, a=1.2, b=0.30):
    """예제 01 에서 회귀로 구한 근사식. 자기 계수로 바꿔 쓰세요.

    EN — The approximation fitted by regression in Example 01. Swap in your own
    coefficients.
    """
    한글 = 한글수(text)
    return 절편 + a * 한글 + b * (len(text) - 한글)


print("추정:", round(estimate_tokens("두 그룹의 평균을 비교하시오."), 1), "토큰")

---
## 1. 입력과 출력은 같은 방을 나눠 쓴다 · Input and output share one room

컨텍스트 윈도우는 **입력만의 상한이 아닙니다.** 입력과 출력을 합친 상한입니다.

```
┌──────────── 컨텍스트 윈도우 ────────────┐
│ system │ 대화 이력 │ 이번 질문 │ 응답 자리 │
│ system │ history   │ this ask  │ room for  │
│        │           │           │ the reply │
└─────────────────────────────────────────┘
```

응답이 들어갈 자리를 남겨 두지 않으면, **보내는 데는 성공하고 답은 못 받습니다.**

먼저 `max_tokens` 가 무엇을 하는지 정확히 봅시다.

> **EN** — The context window is **not a cap on the input alone**; it caps input
> and output together. Leave no room for the reply and you succeed in sending
> while failing to receive. First, let's see exactly what `max_tokens` does.

In [ ]:
질문 = "분산분석(ANOVA)이 무엇인지 설명해 주세요."

for m in [20, 60, 200, 600]:
    r = raw([{"role": "user", "content": 질문}], max_tokens=m)
    답 = r.choices[0].message.content.strip()
    print(f"max_tokens={m:4d}  실제출력={r.usage.completion_tokens:4d}  "
          f"finish_reason={r.choices[0].finish_reason:8s}")
    print(f"   ...{답[-40:]!r}")
    print()
    time.sleep(1)

### 무엇이 보이는가 · What you should see

- `max_tokens` 는 **상한일 뿐**입니다. 할 말이 끝나면 그전에 멈춥니다.
  600 으로 올려도 출력이 600 이 되지 않습니다.
- `finish_reason` 이 `"length"` 면 **잘린 것**이고 `"stop"` 이면 정상 종료입니다.
- 잘린 응답은 문장 중간에서 끝납니다. 위 출력의 마지막 40자를 보십시오.

**항상 `finish_reason` 을 확인하십시오.** 이 한 줄이 디버깅 시간을 몇 시간
아껴 줍니다.

> **EN**
> - `max_tokens` is **only a ceiling**: the model stops earlier when it is done,
>   so raising it to 600 does not make the output 600 tokens long.
> - `finish_reason == "length"` means **it was cut off**; `"stop"` means it
>   finished normally.
> - A truncated answer ends mid-sentence — look at the last 40 characters above.
>
> **Always check `finish_reason`.** That one line saves hours of debugging.

---
## 2. 잘린 응답을 파싱하면 · What happens when you parse a truncated answer

3주차에 우리는 모델에게 JSON 을 받게 됩니다. 그때 `max_tokens` 가 모자라면
어떤 일이 일어나는지 미리 보겠습니다.

> **EN** — In week 3 we start receiving JSON from the model. Here is a preview of
> what happens then if `max_tokens` is too small.

In [ ]:
import json

프롬프트 = ('아래 변수들을 연속형과 범주형으로 분류해 JSON 으로만 답하세요.\n'
          '변수: 나이, 성별, 혈압, 지역, 체질량지수, 흡연여부, 소득, 학력\n'
          '형식: {"연속형": [...], "범주형": [...]}')

for m in [25, 300]:
    r = raw([{"role": "user", "content": 프롬프트}], max_tokens=m)
    본문 = r.choices[0].message.content.strip()
    이유 = r.choices[0].finish_reason
    print(f"--- max_tokens={m}  finish_reason={이유} ---")
    print(본문)
    try:
        json.loads(본문)
        print("파싱: 성공")
    except json.JSONDecodeError as e:
        print(f"파싱: 실패 — {e}")
    print()
    time.sleep(1)

짧은 쪽은 파싱에 실패합니다. 그런데 **모델이 JSON 을 못 만든 것이 아닙니다.**
우리가 잘랐을 뿐입니다.

`finish_reason` 을 안 보면 "이 모델은 JSON 을 못 만든다" 는 잘못된 결론에
이릅니다. 3주차에서 이 구분이 중요해집니다.

> **EN** — The short one fails to parse — but **the model did not fail to produce
> JSON**; we truncated it. Without looking at `finish_reason` you reach the wrong
> conclusion, that "this model cannot produce JSON". That distinction matters in
> week 3.

---
## 3. 한계까지 밀어 보기 · Pushing to the limit

이제 입력을 점점 늘려 가며 어디서 무엇이 무너지는지 봅니다.

> 서버에 부담이 가는 구간입니다. 대기 시간을 지우지 마세요.
> 실패가 나오면 그 시점에서 멈춥니다.

> **EN** — Now we grow the input step by step and watch where things break.
> This section is hard on the server: do not remove the waits. The loop stops at
> the first failure.

In [ ]:
씨앗 = ("통계 분석에서 표본의 대표성은 매우 중요한 문제이다. "
       "표본이 모집단을 제대로 반영하지 못하면 어떤 정교한 방법을 쓰더라도 "
       "결론을 일반화할 수 없다. ")


def 더미(목표토큰):
    """대략 목표 토큰만큼의 한국어 텍스트를 만든다.

    EN — Build roughly the requested number of tokens' worth of Korean text.
    """
    글 = ""
    while estimate_tokens(글) < 목표토큰:
        글 += 씨앗
    return 글


결과 = []
for 목표 in [500, 1000, 2000, 4000, 8000, 16000]:
    본문 = 더미(목표)
    추정 = estimate_tokens(본문)
    try:
        r = raw([{"role": "user", "content": 본문 + "\n\n위 글을 한 문장으로 요약하세요."}],
                max_tokens=100)
        결과.append({"목표": 목표, "추정입력": round(추정),
                    "실제입력": r.usage.prompt_tokens,
                    "출력": r.usage.completion_tokens,
                    "finish_reason": r.choices[0].finish_reason,
                    "상태": "성공"})
        print(f"[{목표:6d}] 성공  실제입력 {r.usage.prompt_tokens:6d}  "
              f"출력 {r.usage.completion_tokens:3d}  {r.choices[0].finish_reason}")
    except Exception as e:
        종류 = type(e).__name__
        결과.append({"목표": 목표, "추정입력": round(추정), "실제입력": None,
                    "출력": None, "finish_reason": None, "상태": f"{종류}"})
        print(f"[{목표:6d}] 실패  {종류}: {str(e)[:110]}")
        break
    time.sleep(2)

In [ ]:
import pandas as pd

한계표 = pd.DataFrame(결과)
한계표["추정오차"] = 한계표["실제입력"] - 한계표["추정입력"]
한계표

### 어떻게 실패했는가 · How did it fail?

실패 방식은 네 가지 중 하나입니다.

| 증상 | 뜻 |
|---|---|
| 명시적 오류 (400 계열) | 가장 친절한 실패다 |
| 조용한 절단 | 앞부분이 잘려 나간다 — `system` 지시가 사라진다 |
| 빈 응답 | 보낼 자리는 있었으나 답할 자리가 없었다 |
| 품질 저하 | 오류 없이 그냥 나빠진다 — 가장 발견하기 어렵다 |

**EN**

| Symptom | What it means |
|---|---|
| An explicit error (4xx) | The kindest failure there is |
| Silent truncation | The front is cut away — the `system` instruction disappears |
| Empty response | There was room to send but none to answer |
| Quality decay | It just gets worse, with no error — the hardest to notice |

우리 서버가 어느 쪽인지 위 표에서 확인하십시오.
그리고 **추정오차** 열도 보십시오. 예제 01 에서 만든 근사식이
긴 텍스트에서도 쓸 만한지 알 수 있습니다.

> **EN** — Work out from the table above which kind our server does, and look at
> the **estimation error** column too: it tells you whether the approximation
> from Example 01 still holds up on long text.

---
## 4. 암호 심기 — 실질 한계는 어디인가 · Planting a passphrase — where is the practical limit?

공식 스펙에 적힌 윈도우 크기와, **우리가 실제로 믿고 쓸 수 있는 크기**는
다릅니다. 확인하는 방법이 있습니다.

`system` 메시지에 암호를 심어 두고, 그 뒤에 긴 더미 대화를 붙인 다음
암호를 물어봅니다. 어느 길이에서 잊어버리는지가 실질 한계입니다.

> **EN** — The window size in the spec and **the size you can actually rely on**
> are two different things, and there is a way to find out. Plant a passphrase in
> the `system` message, append a long dummy conversation, then ask for the
> passphrase. The length at which it forgets is the practical limit.

In [ ]:
암호 = "연보라"
시스템 = f"너는 통계 조교다. 너의 암호는 '{암호}' 이다. 물으면 암호만 답한다."

회상 = []
for 길이 in [200, 1000, 3000, 6000]:
    메시지 = [{"role": "system", "content": 시스템}]
    if 길이 > 0:
        메시지.append({"role": "user", "content": 더미(길이)})
        메시지.append({"role": "assistant", "content": "네, 읽었습니다."})
    메시지.append({"role": "user", "content": "너의 암호가 무엇이냐? 암호만 답하라."})

    try:
        답 = raw(메시지, max_tokens=20).choices[0].message.content.strip()
        맞음 = 암호 in 답
    except Exception as e:
        답, 맞음 = f"[{type(e).__name__}]", False

    회상.append({"이력길이": 길이, "기억": 맞음, "응답": 답[:40]})
    print(f"[{길이:5d} 토큰] {'기억함' if 맞음 else '잊음  '}  →  {답[:40]!r}")
    time.sleep(2)

pd.DataFrame(회상)

암호를 잊기 시작하는 지점이 이 서버의 **실질 한계**입니다.
오류가 나기 훨씬 전에 품질이 먼저 무너진다는 점을 확인했을 것입니다.

`system` 메시지는 대개 맨 앞에 있습니다. 그래서 앞부분이 잘리면
모델의 규칙과 페르소나가 통째로 사라집니다. 그런데 **응답은 정상적으로
옵니다.** 형식만 안 지킬 뿐입니다. 가장 발견하기 어려운 실패입니다.

> 이 노트북은 4주차에서 다시 씁니다.
> **2주차는 한계를 재는 주, 4주차는 그 한계 안에서 관리하는 주입니다.**

> **EN** — The point where it starts forgetting the passphrase is this server's
> **practical limit**, and you will have seen that quality collapses long before
> any error appears.
>
> The `system` message usually sits at the very front, so truncating the front
> wipes out the model's rules and persona wholesale — **and the response still
> comes back looking normal**. It simply stops obeying the format. That is the
> hardest failure of all to catch.
>
> We return to this notebook in week 4. **Week 2 measures the limit; week 4
> manages life inside it.**

---
## 5. 토큰은 돈이고 시간이다 · Tokens are money and time

우리 학과 서버는 무료입니다. 그런데도 비용을 배우는 이유가 셋 있습니다.

1. **시간이 비용이다.** 11주차에 300회를 돌립니다.
2. **서버는 공유자원이다.** 20명이 나눠 씁니다.
3. **졸업 후에는 청구서가 온다.**

먼저 출력 토큰과 응답 시간의 관계를 재 봅시다.

> **EN** — Our department server is free, yet there are three reasons to learn
> about cost: (1) **time is cost** — week 11 runs 300 calls; (2) **the server is
> a shared resource**, split among 20 people; and (3) **after you graduate, an
> invoice arrives.** First, let's measure how response time relates to output
> tokens.

In [ ]:
지연 = []
for 지시, m in [("한 단어로 답하세요.", 20),
              ("한 문장으로 답하세요.", 60),
              ("한 문단으로 답하세요.", 200),
              ("자세히 설명해 주세요.", 500)]:
    t0 = time.time()
    r = raw([{"role": "user", "content": f"분산분석이 무엇인가요? {지시}"}], max_tokens=m)
    걸린시간 = time.time() - t0
    지연.append({"지시": 지시, "출력토큰": r.usage.completion_tokens,
                "초": round(걸린시간, 2)})
    print(f"{지시:20s} 출력 {r.usage.completion_tokens:4d} 토큰  {걸린시간:5.2f} 초")
    time.sleep(1)

표 = pd.DataFrame(지연)
표["토큰당초"] = (표["초"] / 표["출력토큰"]).round(4)
표

응답 시간은 대체로 **출력 토큰 수에 비례**합니다.
그래서 결론은 하나입니다. **출력을 줄여라.**

`"간결하게 답하라"` 한 줄이 시간과 비용을 절반으로 줄이기도 합니다.

> **EN** — Response time is broadly **proportional to the number of output
> tokens**, so there is one conclusion: **shorten the output.** A single line
> saying "answer concisely" can halve both time and cost.

In [ ]:
# 상용 API 를 쓴다면 얼마일까 — 단가는 예시입니다.
# EN — What would a commercial API cost? The unit prices are illustrative.
입력단가 = 0.15 / 1_000_000     # USD per token (예시) / USD per token (example)
출력단가 = 0.60 / 1_000_000     # 출력이 보통 3~5배 비싸다 / output usually costs 3-5x more

def 비용(입력토큰, 출력토큰):
    return 입력토큰 * 입력단가 + 출력토큰 * 출력단가


시나리오 = [
    ("한 번 호출",      1_000,     300),
    ("에이전트 한 판",   12_000,   2_000),
    ("평가 한 번",   1_200_000, 200_000),
]
for 이름, i, o in 시나리오:
    print(f"{이름:14s} 입력 {i:9,d} + 출력 {o:7,d} 토큰   →   ${비용(i, o):8.4f}")

---
## 6. 에이전트가 비싼 진짜 이유 · The real reason agents are expensive

에이전트는 루프를 돕니다. 그리고 매 스텝마다 **지금까지의 이력 전체**를
다시 보냅니다.

```
1스텝: 1t    2스텝: 2t    3스텝: 3t   ...   n스텝: n·t

누적 = t(1 + 2 + ... + n) = t · n(n+1)/2   →   O(n²)
cumulative = t(1 + 2 + ... + n) = t · n(n+1)/2   →   O(n²)
```

등차수열의 합입니다. 계산해 보면 실감이 다릅니다.

> **EN** — An agent loops, and at every step it resends **the entire history so
> far**. That is the sum of an arithmetic series — and working the numbers makes
> it feel very different from reading about it.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

스텝당 = 800          # 스텝 하나가 이력에 더하는 토큰 / tokens one step adds to the history
n = np.arange(1, 13)

누적_압축없음 = 스텝당 * n * (n + 1) / 2
누적_압축있음 = 스텝당 * n * 3          # 최근 3스텝만 유지한다고 가정 / assume we keep the last 3 steps

for i in [3, 6, 9, 12]:
    print(f"{i:2d}스텝   압축 없음 {누적_압축없음[i-1]:8,.0f}   "
          f"압축 있음 {누적_압축있음[i-1]:8,.0f}   "
          f"배율 {누적_압축없음[i-1] / 누적_압축있음[i-1]:.1f}배")

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(n, 누적_압축없음, "o-", label="no compression  O(n^2)")
ax.plot(n, 누적_압축있음, "s-", label="keep last 3     O(n)")
ax.set_xlabel("loop steps")
ax.set_ylabel("cumulative input tokens")
ax.set_title("Why an agent loop gets expensive")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

6스텝이면 한 번 호출의 21배입니다. 스텝을 6에서 10으로 늘리면
비용은 21에서 55로, **2.6배**가 됩니다.

에이전트를 조금 더 똑똑하게 만들려고 루프를 몇 바퀴 더 돌리면
비용은 몇 배가 아니라 제곱으로 늘어납니다.

> 6주차에서 스텝 상한을 반드시 두라고 하는 이유가 이것입니다.
> 안전 문제이기도 하지만 **비용 문제이기도** 합니다.

> **EN** — Six steps costs 21 times a single call. Going from 6 steps to 10
> takes the cost from 21 to 55 — **2.6 times more**. Spinning the loop a few more
> turns to make the agent slightly smarter multiplies the cost quadratically, not
> linearly.
>
> This is why week 6 insists on a step ceiling. It is a safety matter, but it is
> **just as much a cost matter**.

---
## 7. 예측과 실측을 겹쳐 본다 · Overlaying prediction and measurement

이론 곡선만 그리고 끝내지 않습니다. 실제 에이전트를 돌려 재 봅니다.

1주차에 만든 숫자 맞히기 에이전트를 20줄로 다시 씁니다.
`usage` 를 누적하는 세 줄만 더했습니다.

> **EN** — We do not stop at drawing the theoretical curve; we run a real agent
> and measure. Here is the week-1 number-guessing agent rewritten in 20 lines,
> with three extra lines that accumulate `usage`.

In [ ]:
import random
import re

def 숫자맞히기(정답, 최대스텝=8):
    """1주차 미니 에이전트 + 토큰 누적.

    EN — The week-1 mini agent, with token accounting added.
    """
    기록, 누적입력, 누적출력, 스텝별 = [], 0, 0, []
    추측, 스텝 = None, 0

    for 스텝 in range(1, 최대스텝 + 1):
        지금까지 = "\n".join(기록) if 기록 else "(아직 없음)"
        프롬프트 = (f"1부터 100 사이 숫자를 맞히는 중이다.\n"
                  f"지금까지의 기록:\n{지금까지}\n"
                  f"다음 추측을 숫자만 답하라.")
        r = raw([{"role": "user", "content": 프롬프트}], max_tokens=10)
        누적입력 += r.usage.prompt_tokens
        누적출력 += r.usage.completion_tokens
        스텝별.append(누적입력 + 누적출력)

        m = re.search(r"\d+", r.choices[0].message.content)
        if not m:
            기록.append("형식 오류 — 숫자만 답하라")
            continue
        추측 = int(m.group())
        if 추측 == 정답:
            기록.append(f"{추측} → 정답")
            break
        기록.append(f"{추측} → {'더 큼' if 추측 < 정답 else '더 작음'}")

    return {"스텝수": 스텝, "누적토큰": 누적입력 + 누적출력,
            "성공": 추측 == 정답, "스텝별": 스텝별}


random.seed(0)
판 = []
for i in range(5):
    결과1 = 숫자맞히기(random.randint(1, 100))
    판.append(결과1)
    print(f"[{i+1}판] {결과1['스텝수']}스텝  누적 {결과1['누적토큰']:5d} 토큰  "
          f"{'성공' if 결과1['성공'] else '실패'}")
    time.sleep(1)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))

for i, r in enumerate(판):
    x = np.arange(1, len(r["스텝별"]) + 1)
    ax.plot(x, r["스텝별"], "o-", alpha=0.7, label=f"run {i+1}")

# 이론 곡선 — 첫 판의 스텝당 토큰으로 맞춘 O(n^2)
# EN — the theoretical curve: O(n^2) scaled to the first run's per-step tokens
첫판 = 판[0]["스텝별"]
t = 첫판[0]
n2 = np.arange(1, max(len(r["스텝별"]) for r in 판) + 1)
ax.plot(n2, t * n2 * (n2 + 1) / 2, "k--", lw=2, label="predicted O(n^2)")

ax.set_xlabel("loop steps")
ax.set_ylabel("cumulative tokens")
ax.set_title("Predicted versus measured")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### 어긋난다면 왜인가 · If they disagree, why?

실측이 예측보다 큰 경우가 대부분입니다. 이유가 둘 있습니다.

- 메시지 **포장 비용**이 스텝마다 붙는다 (예제 01 의 절편).
- 기록이 쌓이면서 프롬프트에 붙는 텍스트가 스텝마다 조금씩 길어진다.

어긋난 이유를 설명할 수 있으면 모형을 이해한 것입니다.

> **EN** — The measurement usually exceeds the prediction, for two reasons: the
> message **packaging cost** is charged at every step (the intercept from Example
> 01), and as the history accumulates the text appended to the prompt grows a
> little each step. If you can explain the discrepancy, you have understood the
> model.

---
## 8. 예산을 코드로 강제한다 · Enforcing the budget in code

"토큰을 아껴 쓰자" 는 지켜지지 않습니다. **넘으면 예외를 던지는 코드**는
지켜집니다.

규율은 문서가 아니라 코드로 세웁니다.

> **EN** — "Let's be careful with tokens" is never obeyed. **Code that raises
> when you go over** is obeyed. Discipline is established in code, not in
> documents.

In [ ]:
class BudgetExceeded(RuntimeError):
    # EN — Raised when the token budget is exhausted.
    pass


class TokenBudget:
    """호출할 때마다 usage 를 기록하고, 상한을 넘으면 멈춘다.

    EN — Record `usage` on every call and stop once the ceiling is crossed.
    """

    def __init__(self, 상한):
        self.상한 = 상한
        self.누적 = 0
        self.호출수 = 0

    def 기록(self, usage):
        # EN — Add this call's usage; raise if it takes us past the ceiling.
        self.누적 += usage.total_tokens
        self.호출수 += 1
        if self.누적 > self.상한:
            raise BudgetExceeded(
                f"토큰 예산 초과: {self.누적:,} / {self.상한:,} "
                f"({self.호출수}회 호출)")
        return self.누적

    @property
    def 남은(self):
        return max(0, self.상한 - self.누적)

    def __repr__(self):
        비율 = self.누적 / self.상한 if self.상한 else 0
        return (f"<TokenBudget {self.누적:,}/{self.상한:,} "
                f"({비율:.0%}), {self.호출수}회>")


budget = TokenBudget(상한=1500)

try:
    for i in range(10):
        r = raw([{"role": "user", "content": f"{i}번 문항: 표준편차를 설명하세요."}],
                max_tokens=120)
        budget.기록(r.usage)
        print(f"  {i}회차  {budget}")
        time.sleep(0.5)
except BudgetExceeded as e:
    print(f"\n[중단] {e}")

20줄이 안 되는 클래스입니다. 그런데 6주차에서 무한 루프를 도는 에이전트를
잡아 주는 것이 이 20줄입니다.

**방어적 프로그래밍의 원칙** — 나쁜 일이 일어날 수 있으면 반드시 일어납니다.
특히 확률적 시스템에서는 더 그렇습니다. 그러니 나쁜 일이 일어났을 때
조용히 진행되지 않고 **시끄럽게 멈추도록** 만들어 두어야 합니다.

이 클래스는 예제 03 에서 `kmu_llm.py` 에 넣습니다.
`LLM` 클래스가 호출할 때마다 자동으로 기록하게 만들면 됩니다.

> **EN** — Fewer than 20 lines — and in week 6 these are the 20 lines that catch
> an agent stuck in an infinite loop.
>
> **The principle of defensive programming**: if something bad can happen, it
> will, and all the more so in a probabilistic system. So arrange for the bad
> thing to **stop loudly** rather than proceed quietly.
>
> This class goes into `kmu_llm.py` in Example 03 — have the `LLM` class record
> to it automatically on every call.

---
## 확인한 것 · What we confirmed

1. 입력과 출력이 **같은 예산**을 나눠 쓴다. 응답 자리를 남겨 둬야 한다.
2. `max_tokens` 는 상한일 뿐이다. `finish_reason` 을 항상 확인하라.
3. 컨텍스트가 넘치면 오류만 나는 게 아니라 **조용히 나빠진다.**
   공식 윈도우 크기와 실질 한계는 다르다.
4. 응답 시간은 출력 토큰에 비례한다. **출력을 줄여라.**
5. 루프의 누적 토큰은 **O(n²)** 다. 스텝 상한이 곧 비용 상한이다.
6. 규율은 코드로 세운다 — `TokenBudget`.

> **EN**
> 1. Input and output share **one budget**; leave room for the reply
> 2. `max_tokens` is only a ceiling — always check `finish_reason`
> 3. Overflowing the context does not merely raise errors, it **quietly degrades**;
>    the official window size and the practical limit are different numbers
> 4. Response time scales with output tokens — **shorten the output**
> 5. A loop's cumulative tokens are **O(n²)**; the step ceiling IS the cost ceiling
> 6. Discipline is established in code — `TokenBudget`

---
## 연습문제 · Exercises

1. `estimate_tokens()` 의 예측값과 실제 `usage` 의 차이를 문장 30개에서 구하고
   잔차를 그리시오. 패턴이 보이면 어떤 변수가 빠진 것인가?
2. 스텝 수 n 에 대한 누적 토큰 식을 유도하고, 최근 k 스텝만 유지할 때의 식과
   비교하시오. 두 식이 만나는 지점은 어디인가?
3. `system` 메시지를 절반으로 줄였을 때 응답 품질이 떨어지는지 문항 10개로
   확인하시오. 토큰은 얼마나 줄었는가?
4. `TokenBudget` 을 고쳐, 상한의 80% 를 넘으면 예외 대신 경고를 출력하고
   100% 에서 멈추도록 만드시오.

> **EN**
> 1. Take 30 sentences, compute the gap between `estimate_tokens()` and the real
>    `usage`, and plot the residuals. If you see a pattern, which predictor is
>    missing?
> 2. Derive the cumulative-token formula in the number of steps n, and compare it
>    with the formula for keeping only the last k steps. Where do the two meet?
> 3. Halve the `system` message and check across 10 items whether answer quality
>    drops. How many tokens did you save?
> 4. Modify `TokenBudget` so that crossing 80% of the ceiling prints a warning
>    instead of raising, and it stops at 100%.

In [ ]:
# 여기에 직접 작성해 보세요 / Write your own code here

<details>
<summary>해설 보기 · Show the solutions</summary>

**1.** 잔차에 패턴이 보이면 근사식이 설명하지 못하는 구조가 남아 있다는
뜻입니다. 대개 숫자·특수문자의 비율, 그리고 문장 부호의 개수입니다.
설명변수를 추가해 다시 적합시키면 패턴이 줄어듭니다.

**2.** 압축이 없으면 `t·n(n+1)/2`, 최근 k 스텝만 유지하면 대략 `t·k·n`
입니다 (초반 k 스텝은 예외). 두 식이 만나는 곳은 `(n+1)/2 = k`,
즉 `n ≈ 2k − 1` 입니다. 그보다 짧은 루프에서는 압축이 이득이 없습니다.
**압축은 공짜가 아니므로 짧은 루프에는 붙이지 않는 편이 낫습니다.**

**3.** 토큰은 확실히 줄어듭니다. 품질은 문항에 따라 다릅니다.
줄인 `system` 에서 형식 지시가 빠졌다면 파싱 실패가 늘어날 것입니다.
이것이 3주차에서 다룰 **비용과 정확도의 교환**입니다. 감으로 정하지 말고
재서 결정하십시오.

**4.** `기록()` 안에서 두 단계로 나누면 됩니다.

```python
def 기록(self, usage):
    self.누적 += usage.total_tokens
    self.호출수 += 1
    if self.누적 > self.상한:
        raise BudgetExceeded(...)
    if self.누적 > self.상한 * 0.8 and not self._경고함:
        print(f"[경고] 예산의 80% 를 넘었습니다: {self.누적:,}/{self.상한:,}")
        self._경고함 = True
    return self.누적
```

경고를 한 번만 내도록 `_경고함` 플래그를 두는 것이 중요합니다.
매번 경고하면 아무도 안 읽습니다. 13주차의 **경보 피로** 이야기입니다.

---

**EN — 1.** A pattern in the residuals means structure the approximation does not
explain — usually the share of digits and symbols, and the number of punctuation
marks. Add those predictors, refit, and the pattern shrinks.

**EN — 2.** Without compression it is `t·n(n+1)/2`; keeping only the last k steps
gives roughly `t·k·n` (the first k steps aside). They meet where `(n+1)/2 = k`,
i.e. `n ≈ 2k − 1`. For loops shorter than that, compression buys nothing.
**Compression is not free, so do not bolt it onto a short loop.**

**EN — 3.** Tokens certainly drop; quality depends on the item. If the format
instruction was among what you cut from `system`, parse failures will rise. This
is the **cost-versus-accuracy trade** of week 3 — decide it by measurement, not
by feel.

**EN — 4.** Split `기록()` into two stages, as shown above. The `_경고함` flag
matters: warn only once. Warn every time and nobody reads it — the **alarm
fatigue** story of week 13.

</details>